# Day 1 — Worked EDA lab

Synthetic retail orders. Read datasets/README.md before making cleaning decisions. Run cells in order. Revenue totals exclude unknown prices; the valid bulk order remains included.

## 1. Inspect the table

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'datasets/retail_orders.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the downloaded repository.')
raw = pd.read_csv(ROOT / 'datasets/retail_orders.csv')
print(raw.head().to_string(index=False))
print('Rows and columns:', raw.shape)
raw.info()


## 2. Audit the data before changing it

In [ ]:
print('Exact duplicate rows:', raw.duplicated().sum())
print('Missing values:')
print(raw.isna().sum())
print('Non-positive quantities:', raw['units'].le(0).sum())


## 3. Apply documented cleaning rules

In [ ]:
orders = raw.drop_duplicates().copy()
invalid = orders['units'].le(0)
rejected = orders.loc[invalid].copy()
orders = orders.loc[~invalid].copy()
orders['channel'] = orders['channel'].fillna('Unknown')
orders['date'] = pd.to_datetime(orders['date'])
orders['revenue'] = orders['units'] * orders['unit_price']
print('Valid orders:', len(orders))
print('Rejected orders:', len(rejected))
print('Orders with unknown revenue:', orders['revenue'].isna().sum())
print('Recorded revenue, priced orders only (EUR):', orders['revenue'].sum(min_count=1))


## 4. Summarise distributions and channels

In [ ]:
print(orders[['units','unit_price','revenue']].describe())
by_channel = orders.groupby('channel').agg(
    order_count=('order_id','size'),
    priced_orders=('revenue','count'),
    recorded_revenue=('revenue',lambda s: s.sum(min_count=1)),
    mean_order_revenue=('revenue','mean'),
    median_order_revenue=('revenue','median'))
print(by_channel.round(2).to_string())


## 5. Plot distributions and comparisons

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
orders['revenue'].dropna().plot.hist(bins=15, ax=axes[0])
axes[0].set(xlabel='Order revenue (EUR)', ylabel='Order count', title='Priced orders')
by_channel['recorded_revenue'].plot.bar(ax=axes[1], rot=0)
axes[1].set(ylabel='Recorded revenue (EUR)', title='Revenue by channel')
axes[2].scatter(orders['units'], orders['revenue'], alpha=0.6)
axes[2].set(xlabel='Units', ylabel='Order revenue (EUR)', title='Units and revenue')
plt.tight_layout()
plt.show()


## 6. Flag unusual observations for review

In [ ]:
q1, q3 = orders['units'].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
flagged = orders.loc[(orders['units'] < lower) | (orders['units'] > upper)]
print('Review fences:', lower, upper)
print(flagged.to_string(index=False))
print('Pearson correlations:')
print(orders[['units','unit_price','revenue']].corr().round(3))


## Interpretation

The larger revenue total is a descriptive observation, not evidence that the channel causes higher sales. Counts, order values and missingness affect comparisons. An IQR flag requests review; it is not an automatic removal rule. Correlation between units and revenue partly reflects the revenue formula.

Extension: recompute the mean and median with and without the bulk order. Label this as sensitivity analysis and explain what information is lost by excluding it.